# Stock Price Prediction using PyTorch

LSTM, CNN, and Transformer models for 17 commercial banks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")

## Load Data

In [ ]:
data = pd.read_csv("../data_preprocessing/combined_banks_dataset.csv")
data['published_date'] = pd.to_datetime(data['published_date'])
data = data.sort_values(['company_id', 'published_date']).reset_index(drop=True)

print(f"Shape: {data.shape}")
print(f"Banks: {data['company_id'].nunique()}")
print(f"Date range: {data['published_date'].min()} to {data['published_date'].max()}")

## Create Sequences

In [ ]:
def create_sequences_per_company(data, seq_length):
    X, y, company_ids, dates = [], [], [], []
    
    for company in data['company_id'].unique():
        company_data = data[data['company_id'] == company].sort_values('published_date')
        prices = company_data['close'].values
        dates_arr = company_data['published_date'].values
        
        for i in range(seq_length, len(prices)):
            X.append(prices[i-seq_length:i])
            y.append(prices[i])
            company_ids.append(company)
            dates.append(dates_arr[i])
    
    return np.array(X), np.array(y), np.array(company_ids), np.array(dates)

SEQ_LENGTH = 60

X, y, company_ids, dates = create_sequences_per_company(data, SEQ_LENGTH)
print(f"X shape: {X.shape}, y shape: {y.shape}")

## Train-Test Split

In [ ]:
min_date = dates.min()
max_date = dates.max()
cutoff_date = min_date + (max_date - min_date) * 0.8

train_mask = dates < cutoff_date
test_mask = dates >= cutoff_date

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

## Scale Data

In [ ]:
scaler = MinMaxScaler()
scaler.fit(y_train.reshape(-1, 1))

def scale_sequences(X, scaler):
    X_scaled = np.zeros_like(X, dtype=np.float32)
    for i in range(len(X)):
        X_scaled[i] = scaler.transform(X[i].reshape(-1, 1)).flatten()
    return X_scaled

X_train_scaled = scale_sequences(X_train, scaler)
y_train_scaled = scaler.transform(y_train.reshape(-1, 1)).flatten()

X_test_scaled = scale_sequences(X_test, scaler)
y_test_scaled = scaler.transform(y_test.reshape(-1, 1)).flatten()

print(f"Train: {X_train_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")

## Dataset Class

In [ ]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = StockDataset(X_train_scaled, y_train_scaled)
test_dataset = StockDataset(X_test_scaled, y_test_scaled)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## LSTM Model

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=100, num_layers=3, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=dropout, bidirectional=True)
        self.fc1 = nn.Linear(hidden_size * 2, 50)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(50, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out.squeeze()

## CNN Model

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, dropout=0.2):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv1d(1, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(2)
        
        self.conv3 = nn.Conv1d(128, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(2)
        
        self.conv5 = nn.Conv1d(64, 32, kernel_size=3, padding=1)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc1 = nn.Linear(32, 100)
        self.fc2 = nn.Linear(100, 50)
        self.fc3 = nn.Linear(50, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = x.permute(0, 2, 1)
        
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool1(x)
        x = self.dropout(x)
        
        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = self.pool2(x)
        x = self.dropout(x)
        
        x = self.relu(self.conv5(x))
        x = self.global_pool(x)
        x = x.squeeze(-1)
        
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x.squeeze()

## Transformer Model

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=3, dropout=0.2):
        super(TransformerModel, self).__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128, 
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc1 = nn.Linear(d_model, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x.squeeze()

## Training Function

In [ ]:
def train_model(model, train_loader, epochs=100, lr=0.001):
    model = model.to(device)
    criterion = nn.HuberLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': []}
    best_loss = float('inf')
    patience_counter = 0
    
    train_size = int(0.9 * len(train_loader.dataset))
    val_size = len(train_loader.dataset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        train_loader.dataset, [train_size, val_size]
    )
    
    train_loader_split = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader_split:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        
        train_loss /= len(train_loader_split)
        val_loss /= len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        scheduler.step(val_loss)
        
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        if patience_counter >= 15:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    model.load_state_dict(best_model_state)
    return model, history

## Evaluation Function

In [ ]:
def evaluate_model(model, test_loader, scaler, y_test_actual):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            predictions.extend(outputs.cpu().numpy())
    
    predictions = np.array(predictions)
    pred = scaler.inverse_transform(predictions.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(y_test_actual, pred))
    mae = mean_absolute_error(y_test_actual, pred)
    r2 = r2_score(y_test_actual, pred)
    mape = np.mean(np.abs((y_test_actual - pred) / y_test_actual)) * 100
    
    return pred, rmse, mae, r2, mape

## Train LSTM

In [ ]:
print("Training LSTM...")
lstm_model = LSTMModel()
lstm_model, history_lstm = train_model(lstm_model, train_loader)
pred_lstm, rmse_lstm, mae_lstm, r2_lstm, mape_lstm = evaluate_model(
    lstm_model, test_loader, scaler, y_test
)

print(f"\nLSTM Results:")
print(f"  RMSE: {rmse_lstm:.4f}")
print(f"  MAE:  {mae_lstm:.4f}")
print(f"  R2:   {r2_lstm:.4f}")
print(f"  MAPE: {mape_lstm:.2f}%")

## Train CNN

In [ ]:
print("Training CNN...")
cnn_model = CNNModel()
cnn_model, history_cnn = train_model(cnn_model, train_loader)
pred_cnn, rmse_cnn, mae_cnn, r2_cnn, mape_cnn = evaluate_model(
    cnn_model, test_loader, scaler, y_test
)

print(f"\nCNN Results:")
print(f"  RMSE: {rmse_cnn:.4f}")
print(f"  MAE:  {mae_cnn:.4f}")
print(f"  R2:   {r2_cnn:.4f}")
print(f"  MAPE: {mape_cnn:.2f}%")

## Train Transformer

In [ ]:
print("Training Transformer...")
transformer_model = TransformerModel()
transformer_model, history_transformer = train_model(transformer_model, train_loader)
pred_trans, rmse_trans, mae_trans, r2_trans, mape_trans = evaluate_model(
    transformer_model, test_loader, scaler, y_test
)

print(f"\nTransformer Results:")
print(f"  RMSE: {rmse_trans:.4f}")
print(f"  MAE:  {mae_trans:.4f}")
print(f"  R2:   {r2_trans:.4f}")
print(f"  MAPE: {mape_trans:.2f}%")

## Results Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['LSTM', 'CNN', 'Transformer'],
    'RMSE': [rmse_lstm, rmse_cnn, rmse_trans],
    'MAE': [mae_lstm, mae_cnn, mae_trans],
    'R2': [r2_lstm, r2_cnn, r2_trans],
    'MAPE': [mape_lstm, mape_cnn, mape_trans]
}).sort_values('RMSE')

print("\n" + "="*70)
print("MODEL RANKING")
print("="*70)
print(results.to_string(index=False))
print(f"\nBest: {results.iloc[0]['Model']}")

## Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    ('LSTM', pred_lstm),
    ('CNN', pred_cnn),
    ('Transformer', pred_trans)
]

for idx, (name, pred) in enumerate(models):
    axes[idx].plot(y_test[:200], label='Actual', linewidth=2)
    axes[idx].plot(pred[:200], label='Predicted', linewidth=2, alpha=0.7)
    axes[idx].set_title(f'{name}')
    axes[idx].set_xlabel('Sample')
    axes[idx].set_ylabel('Price')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].bar(results['Model'], results['RMSE'])
axes[0,0].set_title('RMSE')
axes[0,0].grid(True, alpha=0.3, axis='y')

axes[0,1].bar(results['Model'], results['MAE'])
axes[0,1].set_title('MAE')
axes[0,1].grid(True, alpha=0.3, axis='y')

axes[1,0].bar(results['Model'], results['R2'])
axes[1,0].set_title('R2')
axes[1,0].grid(True, alpha=0.3, axis='y')

axes[1,1].bar(results['Model'], results['MAPE'])
axes[1,1].set_title('MAPE %')
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

histories = [
    ('LSTM', history_lstm),
    ('CNN', history_cnn),
    ('Transformer', history_transformer)
]

for idx, (name, hist) in enumerate(histories):
    axes[idx].plot(hist['train_loss'], label='Train')
    axes[idx].plot(hist['val_loss'], label='Val')
    axes[idx].set_title(f'{name} Training')
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Loss')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()